The ``dPULearn().mine_negatives()`` method mines reliable negatives from an unlabeled pool in one call. Instead of stacking the positive and unlabeled feature matrices by hand, building a ``1`` (positive) / ``2`` (unlabeled) label vector, fitting, and slicing the mined rows back out by index, you pass the two matrices separately and receive a boolean mask over the rows of ``X_unlabelled``.

In [1]:
import aaanalysis as aa
import numpy as np
import pandas as pd
aa.options["verbose"] = False

# Build a CPP feature matrix for the gamma-secretase PU dataset (substrates vs unlabeled others).
df_seq = aa.load_dataset(name="DOM_GSEC_PU")
df_feat = aa.load_features(name="DOM_GSEC")
sf = aa.SequenceFeature()
X = sf.feature_matrix(features=df_feat["feature"], df_parts=sf.get_df_parts(df_seq=df_seq))
labels = df_seq["label"].to_numpy()

# Split into the positive (1) and unlabeled (2) feature blocks.
X_pos = X[labels == 1]
X_unl = X[labels == 2]
print(f"positives: {len(X_pos)} | unlabeled: {len(X_unl)}")

positives: 63 | unlabeled: 631


Mine a fixed number of reliable negatives directly from the unlabeled pool with ``n_unl_to_neg``. The returned boolean ``mask_neg`` flags, over the rows of ``X_unlabelled``, which unlabeled samples were identified as reliable negatives.

In [2]:
dpul = aa.dPULearn(random_state=42)
mask_neg = dpul.mine_negatives(X_pos=X_pos, X_unlabelled=X_unl, n_unl_to_neg=49)
print(f"mined reliable negatives: {int(mask_neg.sum())} of {len(X_unl)} unlabeled")
X_neg = X_unl[mask_neg]  # the mined feature rows

mined reliable negatives: 49 of 631 unlabeled


The mask equals the manual stacking path exactly: stack ``X_pos`` over ``X_unlabelled``, fit with ``1`` / ``2`` labels, and slice ``labels_[len(X_pos):] == 0``.

In [3]:
# Stack X_pos over X_unlabelled, fit with 1 / 2 labels, and slice labels_[len(X_pos):] == 0.
labels_manual = np.asarray(
    aa.dPULearn(random_state=42)
    .fit(X=np.vstack([X_pos, X_unl]),
         labels=np.array([1] * len(X_pos) + [2] * len(X_unl)),
         n_unl_to_neg=49)
    .labels_)
mask_manual = labels_manual[len(X_pos):] == 0
print("mask equals manual path:", np.array_equal(mask_neg, mask_manual))

mask equals manual path: True


After mining, the instance is fitted: ``labels_`` (over the stacked positives then unlabeled) and ``df_pu_`` are set, so the :class:`dPULearnPlot` methods work as usual. Use ``n_neg`` instead of ``n_unl_to_neg`` to request a total count, or set a distance ``metric`` (``euclidean`` / ``manhattan`` / ``cosine``) for distance-based identification.

In [4]:
print(pd.Series(dpul.labels_).value_counts())
aa.display_df(df=dpul.df_pu_[dpul.df_pu_["selection_via"].str.contains("PC", na=False)],
              n_rows=10, n_cols=5, show_shape=True)

2    582
1     63
0     49
Name: count, dtype: int64
DataFrame shape: (49, 15)


,selection_via,PC1 (56.2%),PC2 (7.4%),PC3 (2.9%),PC4 (2.8%)
84,PC1,0.021000,-0.047800,0.075200,-0.005400
95,PC2,0.032000,-0.082100,0.025800,-0.037700
109,PC1,0.026100,-0.058500,0.075700,-0.020900
158,PC1,0.023500,-0.060700,0.054000,0.000900
161,PC1,0.025900,0.031400,0.044900,0.055400
170,PC1,0.026100,-0.035300,0.058300,0.025800
192,PC6,0.040100,-0.002200,0.004300,-0.053600
193,PC1,0.024700,-0.056900,0.051300,-0.035600
195,PC5,0.029900,0.006500,0.035800,0.050200
200,PC1,0.021200,-0.056200,0.005700,0.072600
